# MongoDB Handling

After installing the MongoDB server in your machine, you can use this notebook for handling the initial processes with the database.

Specifically, in this step, we utilize Python's `pymongo` library to exploit its capabilities for MongoDB server interaction.

**Important Note: Be sure that the MongoDB server is up and running as a service in the background.**

For example, in macOS, to run MongoDB (i.e. the mongod process) as a service, run:

* `brew services start mongodb-community`

To stop a mongod running as a macOS service, use the following command as needed:

* `brew services stop mongodb-community`

To install MongoDB in your system, follow the instructions here:

* https://www.mongodb.com/docs/manual/administration/install-community/


**Note:** You can modify any of the processes below, however, you have to explain your thoughts.

In [1]:
# import library for various processes with the OS
import os

## Load configuration

In [2]:
# import library for yaml handling
import yaml

In [3]:
config_path = os.path.join(os.getcwd(), "config.yml")

with open(config_path) as file:
    config = yaml.load(file, Loader=yaml.FullLoader)

## MongoDB database instantiation

The relevant information for the MongoDB client connection, the database name, and collection name is located in the configuration file.

```
# DB Connection with the uri (host)
client: "mongodb://localhost:27017/"

# db name
db: "aiot_course"

# db collection
col: "NAME YOUR COLLECTION"
```

In [4]:
# import library for hanlding the MongoDB client
import pymongo
# import library for retrieving datetime
from datetime import datetime

### Create the database

To create a database in MongoDB, start by creating a MongoClient object, then specify a connection URL with the correct ip address and the name of the database you want to create.

MongoDB will create the database if it does not exist, and make a connection to it.

In [5]:
client = pymongo.MongoClient(config["client"])

In [6]:
db = client[config["db"]]

### Instantiate the collection

To create a collection in MongoDB, use the database object and specify the name of the collection you want to create.

MongoDB will create the collection if it does not exist.

In [7]:
col = db[config["col"]]

Initially, no collection will be shown in MongoDB before you enter the first document!

## Create the data collection

Uploading the gathered data to MongoDB collection. The data directory structure should be as follows:

```
.
└── data/
    ├──
    ├── scroll-up-thumb/
    │   ├── scroll-up-thumb_0_50_AccGyr_1_1_01_61964f51d11ec26c3bbde60b.csv
    │   ├── scroll-up-thumb_0_50_AccGyr_1_1_01_7982f51d11ec26c3be60b875.csv
    │   └── ..
    ├── scroll-up-down/
    │   ├── scroll-up-down_0_50_AccGyr_1_1_01_12432f51d11ec26c3b944jd0.csv
    │   ├── scroll-up-down_0_50_AccGyr_1_1_02_47563f51d11ec26c3bb4210k.csv
    │   └── .
    └── class ...
```

In [8]:
# import library for handling the csv data and transformations
import pandas as pd
import json

Get data path:

In [9]:
data_path = os.path.join(os.getcwd(), "data")
print(data_path)

/Users/koules/Developer/4th_year/IoT-Course-AIoT-project/data


List all files in a path:

In [10]:
classes_folders_list = [f for f in os.listdir(data_path) if os.path.isdir(os.path.join(data_path, f))]
print(classes_folders_list)

['swipe-right-index', 'scroll-down-index', 'texting-single', 'scroll-up-index', 'swipe-left-index']


In [32]:
for i in range(0, len(files_in_folder), 2):
    print('_'.join(files_in_folder[i].split("_")[:-1]) + f'_0{i//2 + 1}00.csv')


swipe-left-index_0_100_AccGyr_1_1_02_0100.csv
swipe-left-index_0_100_AccGyr_1_1_02_0200.csv
swipe-left-index_0_100_AccGyr_1_1_02_0300.csv
swipe-left-index_0_100_AccGyr_1_1_02_0400.csv
swipe-left-index_0_100_AccGyr_1_1_02_0500.csv


Each document in the MongoDB database should have the following schema:

```json
{
  "_id": ObjectId("6984b3fa87abe7f4dff571aa"),
  "data": {
    "acc_x": ["array", "of", "values"],
    "acc_y": ["array", "of", "values"],
    "acc_z": ["array", "of", "values"]
  },
  "gesture_id": "The label of the instance",
  "hand": 0,
  "sr": 50,
  "sensor": "AccGyr",
  "primary": 1,
  "spontaneous": 1,
  "user": "01",
  "datetime": "MongoDB datetime object (it can be generated with the datetime.datetime.now() function"
}
```

Accordingly, if you are using gyroscope or both accelerometer and gyroscope, the following order and naming of the sensor keys should be defined:

* for gyroscope: `gyr_x`, `gyr_y`, `gyr_z` for the three axes
* for accelerometer and gyroscope: `acc_x`, `acc_y`, `acc_z`, `gyr_x`, `gyr_y`, `gyr_z` for the six axes

**Note: Be careful, the document is mandatory to have the aforementioned schema, in order to argue and proceed with the rest of the processes later on, in data engineering, plotting, etc.**

Now we'll merge the two metrics, they were recorded in parallel using session from the mobile app, somethimes there appears to be a 1ms difference but we dismiss it as trivial and merge accordingly

In [ ]:
# ONLY RUN IF alr merged

# merged_df = pd.read_csv(os.path.join(folder_path, files_in_folder[0]))
# print(merged_df.head())

   Unnamed: 0  acc_x  acc_y  acc_z  gyr_x  gyr_y  gyr_z
0           0  0.242 -0.394  0.886 -6.098  6.768  3.415
1           1  0.236 -0.406  0.896 -5.915  6.829  2.622
2           2  0.262 -0.405  0.910 -4.634  5.671  2.378
3           3  0.250 -0.409  0.907 -2.866  3.537  2.134
4           4  0.234 -0.405  0.907 -2.561  1.098  2.195


In [44]:
# print files in folder
folder_path = os.path.join(data_path, classes_folders_list[0])
# files_in_folder = [f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f)) and os.path.join(folder_path, f).split('_')[-2] == '02']
files_in_folder = [f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f)) and os.path.join(folder_path, f).split('_')[-2] == '02' and os.path.join(folder_path, f).split('_')[-1][1] == '0']
files_in_folder.sort()
print(files_in_folder)

['swipe-right-index_0_100_AccGyr_1_1_02_0011.csv', 'swipe-right-index_0_100_AccGyr_1_1_02_0012.csv', 'swipe-right-index_0_100_AccGyr_1_1_02_0021.csv', 'swipe-right-index_0_100_AccGyr_1_1_02_0022.csv', 'swipe-right-index_0_100_AccGyr_1_1_02_0031.csv', 'swipe-right-index_0_100_AccGyr_1_1_02_0032.csv', 'swipe-right-index_0_100_AccGyr_1_1_02_0041.csv', 'swipe-right-index_0_100_AccGyr_1_1_02_0042.csv', 'swipe-right-index_0_100_AccGyr_1_1_02_0051.csv', 'swipe-right-index_0_100_AccGyr_1_1_02_0052.csv']


In [46]:
# merge acc and gyro metrics in a single file
from utils import df_rebase

to_insert = []

for i in range(0, len(files_in_folder), 2):
    df_acc = pd.read_csv(os.path.join(folder_path, files_in_folder[i])) 
    df_gyr = pd.read_csv(os.path.join(folder_path, files_in_folder[i+1]))

    gyr_cols = df_gyr[['x-axis (deg/s)', 'y-axis (deg/s)', 'z-axis (deg/s)']]
    merged_df = pd.concat([df_acc, gyr_cols], axis=1)

    start = list(merged_df).index('x-axis (g)')
    cols_to_rename = list(merged_df.columns[start:])
    targets = config["rename"]

    merged_df = df_rebase(merged_df, cols_to_rename, targets)
    to_insert.append(merged_df)
        
    output__filename = '_'.join(files_in_folder[i].split("_")[:-1]) + f'_0{i//2 + 1}00.csv'
    # merged_df.to_csv(os.path.join(folder_path, output__filename))

print(to_insert)

# mapper = dict(zip(cols_to_rename, targets))
# # print(mapper)

# merged_df = merged_df.rename(columns=mapper)
# # print(merged_df.columns)

# merged_df.to_csv(os.path.join(folder_path, 'scroll-up-index_1_100_AccGyr_0_1_01_4321.csv'))
# for file in files_in_folder:
#     if (os.path.exists(os.path.join(folder_path, file))):
#         os.remove(os.path.join(folder_path, file))

Initial columns: ['epoc (ms)', 'timestamp (+0300)', 'elapsed (s)', 'x-axis (g)', 'y-axis (g)', 'z-axis (g)', 'x-axis (deg/s)', 'y-axis (deg/s)', 'z-axis (deg/s)']
Processed columns: ['acc_x', 'acc_y', 'acc_z', 'gyr_x', 'gyr_y', 'gyr_z']
Initial columns: ['epoc (ms)', 'timestamp (+0300)', 'elapsed (s)', 'x-axis (g)', 'y-axis (g)', 'z-axis (g)', 'x-axis (deg/s)', 'y-axis (deg/s)', 'z-axis (deg/s)']
Processed columns: ['acc_x', 'acc_y', 'acc_z', 'gyr_x', 'gyr_y', 'gyr_z']
Initial columns: ['epoc (ms)', 'timestamp (+0300)', 'elapsed (s)', 'x-axis (g)', 'y-axis (g)', 'z-axis (g)', 'x-axis (deg/s)', 'y-axis (deg/s)', 'z-axis (deg/s)']
Processed columns: ['acc_x', 'acc_y', 'acc_z', 'gyr_x', 'gyr_y', 'gyr_z']
Initial columns: ['epoc (ms)', 'timestamp (+0300)', 'elapsed (s)', 'x-axis (g)', 'y-axis (g)', 'z-axis (g)', 'x-axis (deg/s)', 'y-axis (deg/s)', 'z-axis (deg/s)']
Processed columns: ['acc_x', 'acc_y', 'acc_z', 'gyr_x', 'gyr_y', 'gyr_z']
Initial columns: ['epoc (ms)', 'timestamp (+0300)', 

## Provide the code to upload the data to MongoDB

In [47]:
for chunk in to_insert:
  result = col.insert_one({
      "data": {
      "acc_x": chunk["acc_x"].tolist(),
      "acc_y": chunk["acc_y"].tolist(),
      "acc_z": chunk["acc_z"].tolist(),
      "gyr_x": chunk["gyr_x"].tolist(),
      "gyr_y": chunk["gyr_y"].tolist(),
      "gyr_z": chunk["gyr_z"].tolist()
    },
    "gesture_id": "swipe-right-index",
    "hand": 0,
    "sr": 100,
    "sensor": "AccGyr",
    "primary": 1,
    "spontaneous": 1,
    "user": "02",
    "datetime": datetime.now()
  })

  print(result.inserted_id)

69ff448e0ac0f73768209d5d
69ff44900ac0f73768209d5e
69ff44930ac0f73768209d5f
69ff44930ac0f73768209d60
69ff44950ac0f73768209d61
